<a href="https://colab.research.google.com/github/JiaHaoNCKU/Automated-Numbering-System/blob/main/main/ANS.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [28]:
!pip install gdspy
!pip install klayout
# 1. 複製 GitHub 倉庫到 Colab 虛擬機
!git clone https://github.com/JiaHaoNCKU/Automated-Numbering-System.git

# 2. 將複製下來的倉庫路徑加入系統搜尋，這樣 import jsonIO 就不會報錯
import sys
sys.path.append('/content/Automated-Numbering-System')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.3/27.3 MB 69.9 MB/s eta 0:00:00
fatal: destination path 'Automated-Numbering-System' already exists and is not an empty directory.


In [24]:
pwd

'/content'

In [25]:
# gds_to_json.py
# ver. 1.0.14
# [Modifications]:
# - Integrated Colab interactive file uploader (files.upload) to let users manually select local GDS files from Win/Mac.
# - Maintained cell parsing hierarchy, layer whitelists, and path routing for module imports.

import os
import sys

# ─── 1. 設定與注入 GitHub 克隆下來的本地虛擬機路徑 ───
GITHUB_REPO_PATH = '/content/Automated-Numbering-System'
MAIN_DIR_PATH = os.path.join(GITHUB_REPO_PATH, 'main')
if MAIN_DIR_PATH not in sys.path:
    sys.path.append(MAIN_DIR_PATH)

# ─── 2. 導入相關模組 ───
import gdspy
import matplotlib.pyplot as plt
from matplotlib.patches import Polygon
import numpy as np
import json
import math
from jsonIO import save_to_json, json_numpy_serializer, load_and_reconstruct_from_json
from google.colab import files  # 🔥 導入 Colab 專用檔案工具

def plot_gds_cell(file_path, cell_name, layers_to_plot=None, show_plot=True, output_png_filename=None, highlight_points=None):
    if not os.path.exists(file_path):
        print(f"❌ Error: File does not exist '{file_path}'")
        return None
    gds_lib = gdspy.GdsLibrary(infile=file_path)
    if cell_name not in gds_lib.cells: return None
    cell_to_plot = gds_lib.cells[cell_name]
    all_polygons = cell_to_plot.get_polygons(by_spec=True)
    fig, ax = plt.subplots(figsize=(12, 12))
    ax.set_aspect('equal', adjustable='box')
    if all_polygons:
        for i, (spec_key, polygons_by_spec_item) in enumerate(all_polygons.items()):
            layer, datatype = spec_key[0], spec_key[1]; layer_key = f"{layer}:{datatype}"
            if layers_to_plot is not None and layer_key not in layers_to_plot: continue
            for points in polygons_by_spec_item:
                ax.add_patch(Polygon(points, closed=True, facecolor='blue', edgecolor='black', linewidth=0.5, alpha=0.7))
    bbox = cell_to_plot.get_bounding_box()
    if bbox is not None:
        ax.set_xlim(bbox[0][0], bbox[1][0]); ax.set_ylim(bbox[0][1], bbox[1][1])
    if output_png_filename: plt.savefig(output_png_filename, dpi=150)
    if show_plot: plt.show()
    return gds_lib

def export_cell_to_recursive_dict(cell, memo=None, layers_to_keep=None):
    if memo is None: memo = {}
    if cell.name in memo: return memo[cell.name]

    print(f"  ⏳ [Processing] Parsing cell geometry structure: {cell.name} ...")

    our_cell_def = {
        "name": cell.name,
        "self_shape": None,
        "subcells": {}
    }
    memo[cell.name] = our_cell_def

    polygons_by_spec = cell.get_polygons(by_spec=True, depth=0)
    for spec_key, polygons_by_spec_item in polygons_by_spec.items():
        layer = spec_key[0]; datatype = spec_key[1]
        if layers_to_keep is not None and (layer, datatype) not in layers_to_keep:
            continue
        layer_key = f"{layer}_{datatype}"
        if len(polygons_by_spec_item) > 0:
            prim_name = f"{cell.name}_merged_polys_{layer_key}"
            poly_prim_def = {
                'name': prim_name,
                'self_shapes': polygons_by_spec_item,
                'subcells': {},
                '_gds_layer': [layer, datatype]
            }
            inst_name = f"{prim_name}_instance"
            our_cell_def['subcells'][inst_name] = {
                'definition': poly_prim_def,
                'instances': [{
                    'origin': np.array([0.0, 0.0]),
                    'rotation': 0.0,
                    'mirror': False,
                    'magnification': 1.0
                }]
            }

    for element in cell.references:
        if isinstance(element, (gdspy.CellReference, gdspy.CellArray)):
            ref_cell = element.ref_cell
            sub_def = export_cell_to_recursive_dict(ref_cell, memo, layers_to_keep)
            instances_list = []

            rot = element.rotation or 0.0
            mirror = bool(element.x_reflection)
            mag = float(element.magnification) if element.magnification else 1.0

            if isinstance(element, gdspy.CellReference):
                instances_list.append({
                    'origin': element.origin,
                    'rotation': rot,
                    'mirror': mirror,
                    'magnification': mag
                })

            elif isinstance(element, gdspy.CellArray):
                origin = element.origin
                cols, rows = element.columns, element.rows
                spacing = element.spacing
                angle_rad = math.radians(rot)

                for r in range(rows):
                    for c in range(cols):
                        x_local = c * spacing[0] * mag
                        y_local = r * spacing[1] * mag

                        if mirror:
                            y_local = -y_local

                        x_rot = x_local * math.cos(angle_rad) - y_local * math.sin(angle_rad)
                        y_rot = x_local * math.sin(angle_rad) + y_local * math.cos(angle_rad)

                        actual_origin = np.array([origin[0] + x_rot, origin[1] + y_rot])
                        instances_list.append({
                            'origin': actual_origin,
                            'rotation': rot,
                            'mirror': mirror,
                            'magnification': mag
                        })

            inst_group_name = f"{ref_cell.name}_instance_group"
            n_conflict = 0
            while inst_group_name in our_cell_def['subcells']:
                n_conflict += 1
                inst_group_name = f"{ref_cell.name}_instance_group_{n_conflict}"

            our_cell_def['subcells'][inst_group_name] = {
                'definition': sub_def,
                'instances': instances_list
            }
    return our_cell_def

def save_cell_to_json(gds_lib, cell_name, output_filename, trace_connection_points=None, layers_to_keep=None):
    target_cell = gds_lib.cells[cell_name]
    recursive_data_structure = export_cell_to_recursive_dict(target_cell, memo={}, layers_to_keep=layers_to_keep)
    save_to_json(recursive_data_structure, output_filename)

if __name__ == "__main__":
    BASE_PROJECT_DIR = '/content/Automated-Numbering-System'
    target_cell_name = "WAFER"

    output_json_path = os.path.join(BASE_PROJECT_DIR, "json", f"{target_cell_name}.json")
    output_png_highlight = os.path.join(BASE_PROJECT_DIR, "resource", f"{target_cell_name}.png")

    os.makedirs(os.path.dirname(output_json_path), exist_ok=True)
    os.makedirs(os.path.dirname(output_png_highlight), exist_ok=True)

    # ─── 3. 🔥 互動式檔案上傳區 ───
    print("👇 請點擊下方按鈕，選擇你電腦本地的 Probes_test.GDS 檔案：")
    uploaded = files.upload()  # 這裡會卡住，並跳出「選擇檔案」的按鈕讓使用者點擊

    # 取得上傳的檔名
    uploaded_gds_name = list(uploaded.keys())[0]
    print(f"✅ 成功載入本地檔案: {uploaded_gds_name}，開始執行幾何轉換...")

    # ─── 4. 執行轉檔 ───
    allowed_layers = [(11, 0), (92, 0), (81, 0), (99, 0), (4, 0), (100, 0)]

    # 讀取剛剛手動上傳到虛擬機根目錄的檔案
    gds_library = gdspy.GdsLibrary(infile=uploaded_gds_name)
    save_cell_to_json(gds_library, target_cell_name, output_json_path, layers_to_keep=allowed_layers)
    print("\n🎉 GDS to JSON conversion completed successfully via manual upload!")

👇 請點擊下方按鈕，選擇你電腦本地的 Probes_test.GDS 檔案：


Saving Probes_test.GDS to Probes_test (1).GDS
✅ 成功載入本地檔案: Probes_test (1).GDS，開始執行幾何轉換...
  ⏳ [Processing] Parsing cell geometry structure: WAFER ...
  ⏳ [Processing] Parsing cell geometry structure: PI32+R-30-S1-L10_102_ZIF ...
  ⏳ [Processing] Parsing cell geometry structure: Probe_32ch_ZIF_Linear_L50 ...
  ⏳ [Processing] Parsing cell geometry structure: Shank_32ch_E25-side ...
  ⏳ [Processing] Parsing cell geometry structure: Electrode_25um ...
  ⏳ [Processing] Parsing cell geometry structure: ARC ...
  ⏳ [Processing] Parsing cell geometry structure: Interconnection_1.2-10_33ch ...
  ⏳ [Processing] Parsing cell geometry structure: Interconnection_7-5_33ch ...
  ⏳ [Processing] Parsing cell geometry structure: ZIF.31+1ch.P500.T ...
  ⏳ [Processing] Parsing cell geometry structure: TEXT$53 ...
  ⏳ [Processing] Parsing cell geometry structure: TEXT ...
  ⏳ [Processing] Parsing cell geometry structure: Cable_32ch_5-5-L5 ...
  ⏳ [Processing] Parsing cell geometry structure: Line5 ...
  ⏳ 

# 新增區段

In [26]:
# probe_hybrid_numbering.py
# ver. 1.0.15
# [Modifications]:
# - Reverted to local virtual machine pathway structure aligned with the GitHub repository directory.
# - Removed interactive file upload buttons to achieve fully automated backend json parsing.
# - Maintained Layer 100 unconditional clearing, multi-level coordinate translation matrices, and big matrix structure tracking.

import numpy as np
import os
import sys
import math

# ─── 1. 設定與注入 GitHub 克隆下來的本地虛擬機路徑 ───
GITHUB_REPO_PATH = '/content/Automated-Numbering-System'
MAIN_DIR_PATH = os.path.join(GITHUB_REPO_PATH, 'main')
if MAIN_DIR_PATH not in sys.path:
    sys.path.append(MAIN_DIR_PATH)

from jsonIO import load_and_reconstruct_from_json, save_to_json

def transform_polygon(poly, origin, rotation_deg, mirror, magnification):
    transformed = []
    for pt in poly:
        x_r = pt[0] * magnification
        y_r = pt[1] * magnification
        if mirror:
            y_r = -y_r
        angle_rad = math.radians(rotation_deg)
        x_g = origin[0] + x_r * math.cos(angle_rad) - y_r * math.sin(angle_rad)
        y_g = origin[1] + x_r * math.sin(angle_rad) + y_r * math.cos(angle_rad)
        transformed.append([x_g, y_g])
    return transformed

def transform_point(point, origin, rotation_deg, mirror=False, magnification=1.0):
    x_r, y_r = point[0] * magnification, point[1] * magnification
    if mirror: y_r = -y_r
    angle_rad = math.radians(rotation_deg)
    x_g = origin[0] + x_r * math.cos(angle_rad) - y_r * math.sin(angle_rad)
    y_g = origin[1] + x_r * math.sin(angle_rad) + y_r * math.cos(angle_rad)
    return [x_g, y_g]

def collect_layer_shapes_recursive(definition, target_layer_num, keys_to_delete=None, cur_rot=0.0, cur_mirror=False, cur_mag=1.0):
    results = []
    def_subcells = definition.get('subcells', {})

    for k, sub_item in list(def_subcells.items()):
        sub_def = sub_item.get('definition', {})
        gds_layer = sub_def.get('_gds_layer')

        if gds_layer and gds_layer[0] == target_layer_num:
            shapes = sub_def.get('self_shapes', [])
            for p in shapes:
                results.append({
                    'poly': p,
                    'rot': cur_rot,
                    'mirror': cur_mirror,
                    'mag': cur_mag,
                    'dict': def_subcells,
                    'key': k
                })
            if keys_to_delete is not None:
                keys_to_delete.append((def_subcells, k))

    for inst_group_name, group_info in def_subcells.items():
        sub_definition = group_info.get('definition', {})
        if sub_definition.get('_gds_layer'):
            continue

        for inst in group_info.get('instances', []):
            origin = inst.get('origin', [0.0, 0.0])
            inst_rot = inst.get('rotation', 0.0)
            inst_mirror = inst.get('mirror', False)
            inst_mag = inst.get('magnification', 1.0)

            next_mag = cur_mag * inst_mag
            next_mirror = cur_mirror ^ inst_mirror
            if cur_mirror:
                next_rot = cur_rot - inst_rot
            else:
                next_rot = cur_rot + inst_rot

            child_results = collect_layer_shapes_recursive(sub_definition, target_layer_num, keys_to_delete, next_rot, next_mirror, next_mag)

            for cr in child_results:
                t_poly = transform_polygon(cr['poly'], origin, inst_rot, inst_mirror, inst_mag)
                results.append({
                    'poly': t_poly,
                    'rot': cr['rot'],
                    'mirror': cr['mirror'],
                    'mag': cr['mag'],
                    'dict': cr['dict'],
                    'key': cr['key']
                })

    return results

def get_digit_polygons(digit, center, size=10.0, gap=0.0):
    x, y = center
    w, h = size * 0.6, size
    t = size * 0.12

    segments = {
        'a': np.array([[gap, h - t], [w - gap, h - t], [w - gap, h], [gap, h]]),
        'b': np.array([[w - t, h / 2 + gap], [w, h / 2 + gap], [w, h - gap], [w - t, h - gap]]),
        'c': np.array([[w - t, gap], [w, gap], [w, h / 2 - gap], [w - t, h / 2 - gap]]),
        'd': np.array([[gap, 0], [w - gap, 0], [w - gap, t], [gap, t]]),
        'e': np.array([[0, gap], [t, gap], [t, h / 2 - gap], [0, h / 2 - gap]]),
        'f': np.array([[0, h / 2 + gap], [t, h / 2 + gap], [t, h - gap], [0, h - gap]]),
        'g': np.array([[gap, h / 2 - t / 2], [w - gap, h / 2 - t / 2], [w - gap, h / 2 + t / 2], [gap, h / 2 + t / 2]])
    }

    digit_map = {
        '0': ['a', 'b', 'c', 'd', 'e', 'f'], '1': ['b', 'c'],
        '2': ['a', 'b', 'g', 'e', 'd'],       '3': ['a', 'b', 'g', 'c', 'd'],
        '4': ['f', 'g', 'b', 'c'],             '5': ['a', 'f', 'g', 'c', 'd'],
        '6': ['a', 'f', 'e', 'd', 'c', 'g'],   '7': ['a', 'b', 'c'],
        '8': ['a', 'b', 'c', 'd', 'e', 'f', 'g'], '9': ['a', 'b', 'c', 'd', 'f', 'g']
    }

    polygons = []
    for seg_key in digit_map.get(str(digit), []):
        poly = segments[seg_key] - np.array([w / 2, h / 2]) + np.array([x, y])
        polygons.append(poly.tolist())
    return polygons

def generate_number_polygons(number_str, center, size=10.0):
    str_num = str(number_str)
    num_digits = len(str_num)
    digit_w = size * 0.6
    spacing_ratio = 0.3
    start_x = center[0] - (digit_w * num_digits + (num_digits - 1) * (digit_w * spacing_ratio)) / 2 + digit_w / 2
    all_polys = []
    for i, char in enumerate(str_num):
        cur_x = start_x + i * (digit_w * (1 + spacing_ratio))
        all_polys.extend(get_digit_polygons(char if char.isdigit() else '1', (cur_x, center[1]), size))
    return all_polys

def process_hybrid_wafer_numbering(json_path, output_json_path):
    print(f"📂 Loading wafer layout JSON: {json_path}")
    data = load_and_reconstruct_from_json(json_path)
    subcells_dict = data.get('subcells', {})
    all_collected_probes = []

    print(f"\n📊 [Debug] Found {len(subcells_dict)} direct subcell groups under top-level WAFER, starting deep recursive sweep...")

    for inst_group_name, group_info in subcells_dict.items():
        definition = group_info.get('definition', {})
        def_name = definition.get('name', 'UNKNOWN')

        if definition.get('_gds_layer'):
            continue

        keys_to_delete_100 = []

        layer_81_results = collect_layer_shapes_recursive(definition, 81)
        layer_100_results = collect_layer_shapes_recursive(definition, 100, keys_to_delete_100)

        if keys_to_delete_100:
            print(f"🧹 [Global Purge] Erasing {len(keys_to_delete_100)} old Layer 100 placeholders from the structural subtree of cell [{def_name}]...")
            for target_dict, k in keys_to_delete_100:
                if k in target_dict:
                    del target_dict[k]

        relative_100_center = None
        local_O_height = 0.0
        base_rot = 0.0
        base_mirror = False
        base_mag = 1.0

        if layer_100_results:
            base_rot = layer_100_results[0]['rot']
            base_mirror = layer_100_results[0]['mirror']
            base_mag = layer_100_results[0]['mag']

            unrotated_verts = []
            for r in layer_100_results:
                for pt in r['poly']:
                    angle_rad = math.radians(-base_rot)
                    x_unrot = pt[0] * math.cos(angle_rad) - pt[1] * math.sin(angle_rad)
                    y_unrot = pt[0] * math.sin(angle_rad) + pt[1] * math.cos(angle_rad)
                    if base_mirror: y_unrot = -y_unrot
                    unrotated_verts.append([x_unrot / base_mag, y_unrot / base_mag])

            concat_verts = np.array(unrotated_verts)
            min_x, min_y = np.min(concat_verts, axis=0)
            max_x, max_y = np.max(concat_verts, axis=0)

            local_O_height = max_y - min_y
            unrotated_center = [(min_x + max_x) / 2.0, (min_y + max_y) / 2.0]

            x_c, y_c = unrotated_center[0] * base_mag, unrotated_center[1] * base_mag
            if base_mirror: y_c = -y_c
            a_rad = math.radians(base_rot)
            relative_100_center = [
                x_c * math.cos(a_rad) - y_c * math.sin(a_rad),
                x_c * math.sin(a_rad) + y_c * math.cos(a_rad)
            ]

        if layer_81_results and relative_100_center is not None:
            probe_type = def_name
            print(f"    ⚡⚡⚡ [Unlock Success] Cell {def_name} meets hybrid positioning conditions! Creating independent serial group for probe model [{probe_type}]")

            for inst in group_info.get('instances', []):
                origin = inst.get('origin', [0.0, 0.0])
                rotation = inst.get('rotation', 0.0)
                mirror = inst.get('mirror', False)
                magnification = inst.get('magnification', 1.0)

                global_center = transform_point(relative_100_center, origin, rotation, mirror, magnification)

                final_mag = magnification * base_mag
                final_mirror = mirror ^ base_mirror
                if mirror:
                    final_rot = rotation - base_rot
                else:
                    final_rot = rotation + base_rot

                all_collected_probes.append({
                    'type': probe_type,
                    'global_center': global_center,
                    'local_size': local_O_height,
                    'final_origin': global_center,
                    'final_rot': final_rot,
                    'final_mirror': final_mirror,
                    'final_mag': final_mag
                })

    grouped_probes = {}
    for probe in all_collected_probes:
        grouped_probes.setdefault(probe['type'], []).append(probe)

    print(f"\n🧩 [Serialization Stage] Injecting new serial numbers directly into the structure of top-level WAFER cell...")
    for p_type, probes_list in grouped_probes.items():
        probes_list.sort(key=lambda p: (-p['global_center'][1], p['global_center'][0]))

        print(f"    ✍️ Assigning independent serial numbers for probe model [{p_type}] (Total: {len(probes_list)} probes)...")
        for idx, probe in enumerate(probes_list):
            serial_number = idx + 1
            display_str = f"{serial_number}"

            digit_polys = generate_number_polygons(display_str, center=[0.0, 0.0], size=probe['local_size'])

            for p_idx, poly_verts in enumerate(digit_polys):
                prim_name = f"wafer_text_{p_type}_{serial_number}_p{p_idx}"
                inst_group_name = f"{prim_name}_instance_group"

                subcells_dict[inst_group_name] = {
                    'definition': {
                        'name': prim_name,
                        'self_shapes': [poly_verts],
                        'subcells': {},
                        '_gds_layer': [100, 0]
                    },
                    'instances': [{
                        'origin': probe['final_origin'],
                        'rotation': probe['final_rot'],
                        'mirror': probe['final_mirror'],
                        'magnification': probe['final_mag']
                    }]
                }
        print(f"    ✅ Successfully completed height-matched and tilt-aligned numbering for model [{p_type}].")

    save_to_json(data, output_json_path)
    print("🎉 Hybrid positioning automated numbering completed successfully.")

if __name__ == "__main__":
    # ─── 2. 統一定義 GitHub 本地倉庫的內建路徑（不需要任何手動上傳按鈕） ───
    BASE_PROJECT_DIR = '/content/Automated-Numbering-System'

    # 讀取與輸出路徑直接在同一個 GitHub 專案的資料夾架構下進行
    in_wafer_json = os.path.join(BASE_PROJECT_DIR, "json", "WAFER.json")
    out_wafer_json = os.path.join(BASE_PROJECT_DIR, "json", "WAFER_numbered.json")

    # 自動建立子資料夾
    os.makedirs(os.path.dirname(out_wafer_json), exist_ok=True)

    # ─── 3. 執行自動化流水號編號 ───
    if os.path.exists(in_wafer_json):
        process_hybrid_wafer_numbering(in_wafer_json, out_wafer_json)
    else:
        print(f"❌ 錯誤：在倉庫路徑下找不到輸入的 JSON 檔案！請確認前一步的輸出路徑是否為：{in_wafer_json}")

📂 Loading wafer layout JSON: /content/Automated-Numbering-System/json/WAFER.json
🔬 Loading and reconstructing data from '/content/Automated-Numbering-System/json/WAFER.json'...
✅ Data loading and reconstruction completed successfully.

📊 [Debug] Found 23 direct subcell groups under top-level WAFER, starting deep recursive sweep...
🧹 [Global Purge] Erasing 1 old Layer 100 placeholders from the structural subtree of cell [PI32+R-30-S1-L10_102_ZIF]...
    ⚡⚡⚡ [Unlock Success] Cell PI32+R-30-S1-L10_102_ZIF meets hybrid positioning conditions! Creating independent serial group for probe model [PI32+R-30-S1-L10_102_ZIF]
🧹 [Global Purge] Erasing 1 old Layer 100 placeholders from the structural subtree of cell [PI16+R-50-S1-L10_65_ZIF_v2]...
    ⚡⚡⚡ [Unlock Success] Cell PI16+R-50-S1-L10_65_ZIF_v2 meets hybrid positioning conditions! Creating independent serial group for probe model [PI16+R-50-S1-L10_65_ZIF_v2]
🧹 [Global Purge] Erasing 1 old Layer 100 placeholders from the structural subtree o

# 新增區段

In [29]:
# json_to_gds.py
# ver. 1.3.6
# [Modifications]:
# - Connected input pathway directly to the GitHub local workspace JSON to remove manual upload triggers.
# - Maintained files.download() to automatically pop up the browser download window for Windows/Mac users.
# - Preserved high-performance cell hierarchy instantiation via KLayout core and zero error checks constraint.

import numpy as np
import json
import re
import sys
import os
import klayout.db as pya
from google.colab import files  # 🔥 用於自動觸發瀏覽器下載檔案

# ─── 1. 設定與注入 GitHub 克隆下來的本地虛擬機路徑 ───
GITHUB_REPO_PATH = '/content/Automated-Numbering-System'
MAIN_DIR_PATH = os.path.join(GITHUB_REPO_PATH, 'main')
if MAIN_DIR_PATH not in sys.path:
    sys.path.append(MAIN_DIR_PATH)

from jsonIO import load_and_reconstruct_from_json

def get_layer_for_name(name, layer_map, layout):
    for regex_str, layer_info in layer_map.items():
        if regex_str == 'default': continue
        if regex_str.startswith('re:'):
            pattern = regex_str[3:]
            if re.search(pattern, name): return layout.layer(layer_info[0], layer_info[1])
        elif name == regex_str: return layout.layer(layer_info[0], layer_info[1])
    default_info = layer_map.get('default', (100, 0))
    return layout.layer(default_info[0], default_info[1])

def create_gds_cell(layout, cell_definition_json, layer_map, existing_cells):
    cell_name = cell_definition_json.get('name', 'UNKNOWN_CELL')
    if cell_name in existing_cells: return existing_cells[cell_name]

    pya_cell = layout.create_cell(cell_name)
    existing_cells[cell_name] = pya_cell

    shape_data = cell_definition_json.get('self_shape')
    shape_list = cell_definition_json.get('self_shapes')

    if shape_data is not None or shape_list is not None:
        original_layer_info = cell_definition_json.get('_gds_layer')
        if original_layer_info and len(original_layer_info) == 2:
            layer = layout.layer(int(original_layer_info[0]), int(original_layer_info[1]))
        else:
            layer = get_layer_for_name(cell_name, layer_map, layout)

        if shape_list is not None:
            for poly_verts in shape_list:
                pya_cell.shapes(layer).insert(pya.DPolygon([pya.DPoint(p[0], p[1]) for p in poly_verts]))
        elif isinstance(shape_data, np.ndarray):
            pya_cell.shapes(layer).insert(pya.DPolygon([pya.DPoint(p[0], p[1]) for p in shape_data]))

    for inst_name, sub_group in cell_definition_json.get('subcells', {}).items():
        sub_cell_def_json = sub_group.get('definition')
        if sub_cell_def_json is None: continue
        pya_sub_cell = create_gds_cell(layout, sub_cell_def_json, layer_map, existing_cells)
        for instance in sub_group.get('instances', []):
            origin = np.array(instance.get('origin', [0.0, 0.0]))
            rotation_deg = instance.get('rotation', 0.0)
            mirror = instance.get('mirror', False)
            mag = instance.get('magnification', 1.0)
            trans = pya.DCplxTrans(mag, rotation_deg, mirror, origin[0], origin[1])
            pya_cell.insert(pya.DCellInstArray(pya_sub_cell.cell_index(), trans))
    return pya_cell

def convert_json_to_gds(input_json_path, output_gds_path, layer_map, top_cell_name="TOP"):
    top_cell_json_data = load_and_reconstruct_from_json(input_json_path)
    layout = pya.Layout()
    layout.dbu = 0.001
    gds_top_cell = layout.create_cell(top_cell_name)
    existing_cells = {}
    pya_main_cell = create_gds_cell(layout, top_cell_json_data, layer_map, existing_cells)
    gds_top_cell.insert(pya.DCellInstArray(pya_main_cell.cell_index(), pya.DCplxTrans(1.0, 0.0, False, 0.0, 0.0)))
    layout.write(output_gds_path)
    print("✅ GDS export completed successfully inside the virtual environment.")

if __name__ == "__main__":
    layer_mapping = {"default": (100, 0)}

    BASE_PROJECT_DIR = '/content/Automated-Numbering-System'

    # ─── 2. 自動對準同一個 GitHub 倉庫下的編號 JSON 檔案 ───
    input_file = os.path.join(BASE_PROJECT_DIR, "json", "WAFER_numbered.json")

    # 定義在虛擬機本地的臨時輸出路徑
    temp_output_gds = "/content/WAFER_numbered2.gds"

    # ─── 3. 執行 KLayout 佈局還原 ───
    if os.path.exists(input_file):
        convert_json_to_gds(input_file, temp_output_gds, layer_mapping, top_cell_name="TOP")

        # ─── 4. 🔥 自動觸發瀏覽器彈出下載視窗，傳送回 Windows/Mac ───
        print("\n💾 正在將還原成功的 GDS 佈局檔案傳送回您的電腦，請留意瀏覽器下載提示...")
        files.download(temp_output_gds)
    else:
        print(f"❌ 錯誤：找不到上一步生成的 JSON 檔案，請確認路徑：{input_file}")

🔬 Loading and reconstructing data from '/content/Automated-Numbering-System/json/WAFER_numbered.json'...
✅ Data loading and reconstruction completed successfully.
✅ GDS export completed successfully inside the virtual environment.

💾 正在將還原成功的 GDS 佈局檔案傳送回您的電腦，請留意瀏覽器下載提示...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>